## Text Summarization

This tutorial follows the text summarization techniques from langchain docs 
You can read more about it [here](https://python.langchain.com/docs/tutorials/summarization/)

### Setup

In [2]:
# setup tracing
import os
from dotenv import load_dotenv
load_dotenv()


os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

os.environ["GROQ_API_KEY" ]  = os.getenv("GROQ_API_KEY")

The central question of building a summarizer is how to pass your documents into the LLM's ocntext windows. Two common approaches are :

1. `Stuff` : Simply "stuff" all your documents into a single prompt. This is the simplest approach.
2. `Map-reduce` : Summarize each doument on its own in a "map" step and then "reduce" the summaries into a final summary.


The second method is effective only when understanding of the sub-document does not rely on preceeding context.
for example, understanding 2nd page doesn't rely on understanding of previous page.

#### Load the Documents

In [3]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs= loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\nAgent System Overview\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final resu

#### Select an LLM

In [5]:
from langchain.chat_models import init_chat_model
from langchain_ollama import ChatOllama

llm = init_chat_model("gemma2-9b-it", model_provider="groq")
# llm = ChatOllama(model="gemma3:4b")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x720707ee0980>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x720707ee0950>, model_name='gemma2-9b-it')

### Method 1 - Stuff: Summarize in a single LLM call

we can use `create_stuff_documents_chain` especially if we are using larger context window models.

This chain will take a list of documents, inserts them all into a prompt, and pass that prompt to an llm

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
loader = PyPDFLoader("apjspeech.pdf")
docs = loader.load()
docs

[Document(metadata={'source': 'apjspeech.pdf', 'page': 0}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealth of our country. During my intera ction at \nRashtrapati Bhavan in Delhi and at every state and union territor y as well as through my \nonline interactions, I have many unique experiences to share with you, which signify the \nfollowi

##

In [8]:
from langchain.chains.summarize import load_summarize_chain
from langchain_core.prompts import ChatPromptTemplate

# Define prompt
prompt = ChatPromptTemplate.from_messages(
    [("system", "Write a concise summary of the following:\\n\\n{context}")]
)


summary_chain= load_summarize_chain(
    llm=llm,
    chain_type="stuff",
    verbose=True
)

output=summary_chain.invoke({"input_documents": docs})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Prompt after formatting:
Write a concise summary of the following:


"A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I have many unique experiences to share with you, which signify the 
following important mes

### Method 2 - Map-Reduce: Summarize long texts via parallelization


we'll first map each document to an individual summary using an LLM. Then we'll reduce or consolidate those summaries into a single global summary.

Map-reduce flows are particularly useful when texts are long compared to the context window of a LLM. For long texts, we need a mechanism that ensures that the context to be summarized in the reduce step does not exceed a model's context window size. Here we implement a recursive "collapsing" of the summaries: the inputs are partitioned based on a token limit, and summaries are generated of the partitions. This step is repeated until the total length of the summaries is within a desired limit, allowing for the summarization of arbitrary-length text.

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate


In [23]:
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=100)
final_documents = splitter.split_documents(docs)
final_documents

[Document(metadata={'source': 'apjspeech.pdf', 'page': 0}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealth of our country. During my intera ction at \nRashtrapati Bhavan in Delhi and at every state and union territor y as well as through my \nonline interactions, I have many unique experiences to share with you, which signify the \nfollowi

In [25]:
chunks_prompt="""
Please summarize the below speech:
Speech:`{text}'
Summary:
"""
map_prompt_template=PromptTemplate(input_variables=['text'],
                                    template=chunks_prompt)

In [26]:
final_prompt='''
Provide the final summary of the entire speech with these important points.
Add a Motivation Title,Start the precise summary with an introduction and provide the summary in number 
points for the speech.
Speech:{text}

'''
final_prompt_template=PromptTemplate(input_variables=['text'],template=final_prompt)
final_prompt_template

PromptTemplate(input_variables=['text'], template='\nProvide the final summary of the entire speech with these important points.\nAdd a Motivation Title,Start the precise summary with an introduction and provide the summary in number \npoints for the speech.\nSpeech:{text}\n\n')

In [ ]:
from langchain.chains.summarize import load_summarize_chain

summary_chain= load_summarize_chain(
    llm=llm,
    chain_type="map_reduce",
    map_prompt=map_prompt_template,
    combine_prompt=final_prompt_template,
    verbose=True
)

output=summary_chain.invoke({"input_documents": final_documents})

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Error in LangChainTracer.on_chain_start callback: ValidationError(model='Run', errors=[{'loc': ('__root__',), 'msg': "argument of type 'NoneType' is not iterable", 'type': 'type_error'}])
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")
Parent run 878620c2-7f47-4e39-83cd-69c1210e3d4f not found for run 38099324-9c1c-41f9-baff-68bf3f8d2486. Treating as a root run.
Parent run 878620c2-7f47-4e39-83cd-69c1210e3d4f not found for run 86e3188e-54ad-45b2-823a-dc551ac3061c. Treating as a root run.
Parent run 878620c2-7f47-4e39-83cd-69c1210e3d4f not found for run dafdac4f-28b3-4dfa-a388-a08da56de809. Treating as a root run.
Parent run 878620c2-7f47-4e39-83cd-69c1210e3d4f not found for run 54bc9f28-fe4a-45e5-bafe-d7f981155b51. Treating as a root run.
Parent run 878620c2-7f47-4e39-83cd-69c1210e3d4f not found for run 0f9d

Prompt after formatting:

Please summarize the below speech:
Speech:`A P J Abdul Kalam Departing speech 
 
 
Friends, I am delighted to address you all, in the country and those livi ng abroad, after 
working with you and completing five beautiful and eventful years in Rashtrapati 
Bhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I 
enjoyed every minute of my tenure enriched by the wonderful assoc iation from each one 
of you, hailing from different walks of life, be it politics, sci ence and technology, 
academics, arts, literature, business, judiciary, administration, local bodies, farming, 
home makers, special children, media and above all from the youth and st udent 
community who are the future wealth of our country. During my intera ction at 
Rashtrapati Bhavan in Delhi and at every state and union territor y as well as through my 
online interactions, I have many unique experiences to share with you, which signify the 
following important mess